# Preprocesamiento

1. Detectar gaps grandes y marcar límites de sesión
2. Remuestrear de segundos a minutos con agregación correcta por tipo de señal
3. Filtrar régimen operacional (Motor_current > umbral)
4. Identificar y numerar sesiones operacionales
5. Identificar regimen 0, 1 y 2
6. Guardar en data/processed/operacional.parquet

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

sns.set_style('whitegrid')

ANALOG_COLS  = ['TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs',
                'Oil_temperature', 'Motor_current', 'Caudal_impulses']
BINARY_COLS  = ['COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS',
                'Pressure_switch', 'Oil_level']

In [36]:
df = pd.read_csv('../data/raw/MetroPT3(AirCompressor).csv',
                parse_dates=['timestamp'])
df.drop(columns=['Unnamed: 0'], inplace=True)
df = df.sort_values('timestamp').reset_index(drop = True)
df.head()

,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
3,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


# 1. Detectar gaps grandes y marcar límites de sesión

#### Verificar fracuencia real

In [45]:
diferencias = df['timestamp'].diff().dt.total_seconds()

print('Frecuencia predominante (segundos):')
print(diferencias.value_counts().head(10))

print("\nGaps mayores a 30 segundos:")
print((diferencias > 30).sum())

print(f'\nGap maximo {diferencias.max():.0f} segundos')
print(f'Gap máximo {diferencias.max()/60:.0f} minutos')
print(f'Gap máximo {diferencias.max()/3600:.0f} horas')

Frecuencia predominante (segundos):
timestamp
10.0    1337521
9.0      128277
12.0      38321
13.0       7988
11.0       4471
21.0         10
19.0          5
22.0          4
20.0          3
17.0          3
Name: count, dtype: int64

Gaps mayores a 30 segundos:
331

Gap maximo 172918 segundos
Gap máximo 2882 minutos
Gap máximo 48 horas
